# Stage 1 Pilot — `planned` variant, seed-controlled LoRA runs

**Kaggle notebook — thin controller only. All logic lives in `scripts/run_pilot_experiment.py`
and `src/training/train_lora.py`.**

Runs the `planned` arm of the Stage 1 pilot comparison (`data-stage-1.md §7`):
3 fresh LoRA fine-tunes of `Qwen/Qwen2.5-Coder-0.5B` on the `planned` FIM
dataset variant, one per seed in `configs/training/pilot.yaml`, all logged to
the shared W&B project. Only the training data and seed vary — same
`configs/training/lora.yaml` hyperparameters as every other run.

Split into one notebook per variant (this one, and its `random.ipynb`
sibling) so each Kaggle session only needs to fit 3 seeds' worth of training,
not all 6 — see `configs/training/pilot.yaml` and `data-stage-1.md §7-8` for
why 2-3 seeds per variant matters (a 10k-file pilot result is a signal, not
proof, and seed variance must be distinguishable from a real distribution
effect).

Data loads directly from Hugging Face (`load_dataset()`), no Kaggle dataset
attachment needed — `data-stage-1.md §7`.

## Required Kaggle setup before running
| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 |
| Internet | **ON** |
| Secret: `HF_TOKEN` | Hugging Face **write** token |
| Secret: `WANDB_API_KEY` | W&B API key from https://wandb.ai/authorize |


In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────────
import os
import shutil
import subprocess
import sys

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"✓ Repository ready at {REPO_DIR}")


In [ ]:
# ── Cell 2: Install dependencies (GPU training + tracking) ───────────────────
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
        "peft>=0.20.0",
        "trl>=0.17.0",
        "transformers>=5.0.0",
        "datasets>=3.0.0",
        "pyarrow",
        "pyyaml>=6.0",
        "huggingface_hub>=0.30.0",
        "wandb>=0.19.0",
    ],
    check=True,
)

print("✓ Dependencies installed")


In [ ]:
# ── Cell 3: Authenticate to Hugging Face + W&B ────────────────────────────────
# Both stored as Kaggle Secrets — NEVER hardcode tokens/keys.
# Add them: Kaggle account → Settings → Secrets → Add New Secret
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

print("✓ HF_TOKEN and WANDB_API_KEY loaded from Kaggle Secrets")


In [ ]:
# ── Cell 4: Run the "planned" variant's pilot seeds ─────────────────────────
# Trains one fresh LoRA run per seed in configs/training/pilot.yaml, all from
# the base model (never continuing from a previous run) — see
# scripts/run_pilot_experiment.py. Each run is logged to W&B individually;
# re-running this cell after an interruption will just start fresh runs for
# whichever seeds haven't completed (no resume logic here — unlike the data
# curation stage, training runs are short enough not to need it).
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/run_pilot_experiment.py", "--variant", "planned"],
    check=True,
)


In [ ]:
# ── Cell 5: Where to look next ────────────────────────────────────────────────
print(
    "✓ 'planned' variant pilot runs complete.\n"
    "  View them in the W&B project: "
    "https://wandb.ai/521er1007-national-institute-of-technology-rourkela/qwen-coder-python-fim\n"
    "  (filter by tag='planned')\n\n"
    "  Once BOTH variant notebooks have finished (this one and random.ipynb),\n"
    "  run: python scripts/analyze_pilot_results.py\n"
    "  to aggregate SAFIM pass@1 across seeds and check whether the "
    "random-vs-planned gap\n"
    "  is larger than seed-to-seed noise (data-stage-1.md §8)."
)
